
# 07 — Build Submission

Objectif :

Créer proprement le fichier ZIP final à soumettre sur Codabench.

Contraintes Codabench importantes :

- le fichier doit s'appeler par exemple `my_submission.zip` ;
- `my_agent.py` doit être **à la racine du ZIP** ;
- pas dans un sous-dossier ;
- l'agent doit définir une classe `MyAgent` ;
- l'import attendu est :
  ```python
  from evaluator.base_agent import BaseAgent
  ```
- l'agent ne doit pas apprendre pendant l'évaluation ;
- l'agent doit retourner une action entière dans `[0, 8]`.

Ce notebook fait :

1. détection automatique du dépôt ;
2. sélection de l'agent final ;
3. validation officielle avec `test_agent_validity.py` ;
4. évaluation locale optionnelle ;
5. création du ZIP ;
6. vérification du contenu du ZIP ;
7. test d'extraction dans un dossier temporaire ;
8. génération d'un rapport.


In [43]:

from pathlib import Path
import subprocess
import zipfile
import shutil
import json
import time
import tempfile
import re
import pandas as pd
import numpy as np



# 1. Détection de la racine du dépôt


In [44]:

def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    candidates = [start] + list(start.parents)
    for p in candidates:
        if (p / "src" / "evaluate_submission.py").exists() and (p / "src" / "test_agent_validity.py").exists():
            return p
    raise FileNotFoundError(
        "Impossible de trouver la racine du dépôt. "
        "Lance ce notebook depuis le dossier du repo ou depuis notebooks/."
    )

REPO_ROOT = find_repo_root()
SRC_DIR = REPO_ROOT / "src"
AGENTS_DIR = SRC_DIR / "agents"
RESULTS_DIR = REPO_ROOT / "results"
SUBMISSIONS_DIR = REPO_ROOT / "submissions"

RESULTS_DIR.mkdir(exist_ok=True)
SUBMISSIONS_DIR.mkdir(exist_ok=True)

print("REPO_ROOT      =", REPO_ROOT)
print("SRC_DIR        =", SRC_DIR)
print("AGENTS_DIR     =", AGENTS_DIR)
print("RESULTS_DIR    =", RESULTS_DIR)
print("SUBMISSIONS_DIR=", SUBMISSIONS_DIR)


REPO_ROOT      = /home/onyxia/work/stable_v2_RL_sailing_challenge
SRC_DIR        = /home/onyxia/work/stable_v2_RL_sailing_challenge/src
AGENTS_DIR     = /home/onyxia/work/stable_v2_RL_sailing_challenge/src/agents
RESULTS_DIR    = /home/onyxia/work/stable_v2_RL_sailing_challenge/results
SUBMISSIONS_DIR= /home/onyxia/work/stable_v2_RL_sailing_challenge/submissions



# 2. Sélectionner l'agent final

Par défaut, le notebook cherche :

1. `src/agents/my_agent.py`
2. `src/agents/agent_final_*.py`
3. `src/agents/agent_tune_*.py`

Si `my_agent.py` n'existe pas, il sélectionne le dernier candidat disponible.


In [45]:

def find_candidate_agents():
    candidates = []

    priority = [
        AGENTS_DIR / "my_agent.py",
        AGENTS_DIR / "agent_trained_example.py",
        AGENTS_DIR / "agent_super_naive.py",
    ]

    for p in priority:
        if p.exists():
            candidates.append(p)

    for pattern in ["agent_final_*.py", "agent_tune_*.py"]:
        for p in sorted(AGENTS_DIR.glob(pattern)):
            if p not in candidates:
                candidates.append(p)

    return candidates

candidate_agents = find_candidate_agents()

print("Candidate agents:")
for i, p in enumerate(candidate_agents):
    print(f"{i}: {p.relative_to(REPO_ROOT)}")

if not candidate_agents:
    raise FileNotFoundError("Aucun agent Python trouvé dans src/agents/.")

# Agent final par défaut : my_agent.py si présent, sinon premier candidat.
FINAL_AGENT_PATH = AGENTS_DIR / "my_agent.py" if (AGENTS_DIR / "my_agent.py").exists() else candidate_agents[0]

print("\nSelected FINAL_AGENT_PATH =", FINAL_AGENT_PATH.relative_to(REPO_ROOT))


Candidate agents:
0: src/agents/my_agent.py
1: src/agents/agent_trained_example.py
2: src/agents/agent_super_naive.py
3: src/agents/agent_final_002.py
4: src/agents/agent_final_006.py
5: src/agents/agent_final_010.py
6: src/agents/agent_tune_000.py
7: src/agents/agent_tune_001.py
8: src/agents/agent_tune_002.py
9: src/agents/agent_tune_003.py
10: src/agents/agent_tune_004.py
11: src/agents/agent_tune_005.py
12: src/agents/agent_tune_006.py
13: src/agents/agent_tune_007.py
14: src/agents/agent_tune_008.py
15: src/agents/agent_tune_009.py
16: src/agents/agent_tune_010.py
17: src/agents/agent_tune_011.py

Selected FINAL_AGENT_PATH = src/agents/my_agent.py



# 3. Vérifications statiques du fichier agent

On vérifie les points simples avant de lancer les scripts officiels.


In [46]:

def static_checks(agent_path):
    text = Path(agent_path).read_text(encoding="utf-8")

    checks = {
        "file_exists": Path(agent_path).exists(),
        "defines_MyAgent": "class MyAgent" in text,
        "has_act": "def act" in text,
        "has_reset": "def reset" in text,
        "has_seed": "def seed" in text,
        "mentions_BaseAgent": "BaseAgent" in text,
        "no_training_keywords_obvious": not any(
            kw in text.lower()
            for kw in [
                "fit(",
                ".backward(",
                "optimizer.step",
                "train(",
                "learn(",
            ]
        ),
    }

    return checks

checks = static_checks(FINAL_AGENT_PATH)
display(pd.DataFrame([checks]).T.rename(columns={0: "status"}))

if not all(checks.values()):
    print("Certaines vérifications statiques sont négatives. Inspecte avant de soumettre.")
else:
    print("Static checks OK.")


,status
file_exists,True
defines_MyAgent,True
has_act,True
has_reset,True
has_seed,True
mentions_BaseAgent,True
no_training_keywords_obvious,True


Static checks OK.



# 4. Fonctions robustes d'exécution


In [47]:

def run_command(cmd, cwd=SRC_DIR, timeout=900):
    start = time.time()
    completed = subprocess.run(
        cmd,
        cwd=str(cwd),
        text=True,
        capture_output=True,
        timeout=timeout,
    )
    elapsed = time.time() - start
    return {
        "cmd": " ".join(str(x) for x in cmd),
        "returncode": completed.returncode,
        "stdout": completed.stdout,
        "stderr": completed.stderr,
        "elapsed_sec": elapsed,
    }

def print_command_result(res, max_chars=4000):
    print("CMD:", res["cmd"])
    print("RETURN CODE:", res["returncode"])
    print("ELAPSED:", round(res["elapsed_sec"], 2), "sec")

    print("\nSTDOUT:")
    print(res["stdout"][:max_chars])
    if len(res["stdout"]) > max_chars:
        print("... [stdout truncated]")

    if res["stderr"]:
        print("\nSTDERR:")
        print(res["stderr"][:max_chars])
        if len(res["stderr"]) > max_chars:
            print("... [stderr truncated]")



# 5. Validation officielle de l'agent

Cette étape doit impérativement passer avant de zipper.


In [48]:

validity_res = run_command(
    ["python", "test_agent_validity.py", str(FINAL_AGENT_PATH)],
    cwd=SRC_DIR,
    timeout=180,
)

print_command_result(validity_res)

validity_ok = validity_res["returncode"] == 0
print("\nVALIDITY OK =", validity_ok)

if not validity_ok:
    raise RuntimeError("L'agent ne passe pas test_agent_validity.py. Corrige avant de créer le ZIP.")


CMD: python test_agent_validity.py /home/onyxia/work/stable_v2_RL_sailing_challenge/src/agents/my_agent.py
RETURN CODE: 0
ELAPSED: 2.81 sec

STDOUT:
✨ Validating agent in: /home/onyxia/work/stable_v2_RL_sailing_challenge/src/agents/my_agent.py ✨

🤖 Agent: MyAgent

✅ SUCCESS: Your agent meets all requirements!

⚠️ Warnings (not required to fix, but recommended):
  1. Agent does not implement save() method
  2. Agent does not implement load() method


🎉 Your agent is ready for submission!
Run this command to test performance before submitting:
python src/evaluate_submission.py /home/onyxia/work/stable_v2_RL_sailing_challenge/src/agents/my_agent.py --seeds 1 --num-seeds 10


VALIDITY OK = True



# 6. Évaluation locale optionnelle

Cette cellule peut être longue.

Pour aller vite :
- mets `RUN_LOCAL_EVAL = False`.

Pour une vraie validation :
- mets `RUN_LOCAL_EVAL = True`,
- `NUM_SEEDS = 50`.


In [49]:

RUN_LOCAL_EVAL = True
NUM_SEEDS = 20
START_SEED = 1

RESULT_RE = re.compile(
    r"(?P<scenario>training_\\d+|test)\\s*(?:\\(TEST\\))?\\s*\\|\\s*"
    r"Success:\\s*(?P<success>[0-9.]+)%\\s*\\|\\s*"
    r"Reward:\\s*(?P<reward>-?[0-9.]+)\\s*±\\s*(?P<std_reward>[0-9.]+)\\s*\\|\\s*"
    r"Steps:\\s*(?P<steps>[0-9.]+)\\s*±\\s*(?P<std_steps>[0-9.]+)"
)

def parse_eval_output(text):
    rows = []
    for line in text.splitlines():
        m = RESULT_RE.search(line)
        if m:
            d = m.groupdict()
            rows.append({
                "scenario": d["scenario"],
                "success_rate": float(d["success"]) / 100.0,
                "mean_reward": float(d["reward"]),
                "std_reward": float(d["std_reward"]),
                "mean_steps": float(d["steps"]),
                "std_steps": float(d["std_steps"]),
            })
    return rows

eval_rows = []
eval_res = None

if RUN_LOCAL_EVAL:
    eval_res = run_command(
        [
            "python", "evaluate_submission.py", str(FINAL_AGENT_PATH),
            "--seeds", str(START_SEED),
            "--num-seeds", str(NUM_SEEDS),
        ],
        cwd=SRC_DIR,
        timeout=1800,
    )

    print_command_result(eval_res, max_chars=8000)

    eval_rows = parse_eval_output(eval_res["stdout"])
    eval_df = pd.DataFrame(eval_rows)

    if len(eval_df) > 0:
        display(eval_df)

        summary = {
            "avg_reward": float(eval_df["mean_reward"].mean()),
            "min_reward": float(eval_df["mean_reward"].min()),
            "avg_success": float(eval_df["success_rate"].mean()),
            "min_success": float(eval_df["success_rate"].min()),
            "avg_steps": float(eval_df["mean_steps"].mean()),
            "max_steps": float(eval_df["mean_steps"].max()),
        }
        print("\nSummary:")
        print(json.dumps(summary, indent=2))
    else:
        print("Parsing failed. Inspect raw output above.")
else:
    print("Local evaluation skipped.")


CMD: python evaluate_submission.py /home/onyxia/work/stable_v2_RL_sailing_challenge/src/agents/my_agent.py --seeds 1 --num-seeds 20
RETURN CODE: 0
ELAPSED: 323.47 sec

STDOUT:
Loaded agent: MyAgent

Evaluating on 3 wind scenarios with 20 seeds
Agent: MyAgent
Maximum steps per episode: 500

WIND_SCENARIO    | SUCCESS RATE | MEAN REWARD       | MEAN STEPS
---------------------------------------------------------------------------
training_1   | Success: 100.00% | Reward: 72.80 ± 1.85 | Steps: 64.4 ± 5.1
training_2   | Success: 100.00% | Reward: 80.09 ± 0.26 | Steps: 45.3 ± 0.6
training_3   | Success: 100.00% | Reward: 71.33 ± 2.27 | Steps: 68.5 ± 6.0
---------------------------------------------------------------------------
OVERALL      | Success: 100.00% ± 0.00%
Reward: 74.74 ± 3.83
Steps: 59.4 ± 10.1

Parsing failed. Inspect raw output above.



# 7. Copier l'agent sous le nom `my_agent.py`

Codabench attend un fichier Python à zipper.

Même si ton agent final s'appelle `agent_final_003.py`, on le copie ici sous :

```text
submissions/my_agent.py
```


In [50]:

SUBMISSION_AGENT = SUBMISSIONS_DIR / "my_agent.py"

shutil.copy2(FINAL_AGENT_PATH, SUBMISSION_AGENT)

print("Copied:")
print(FINAL_AGENT_PATH.relative_to(REPO_ROOT), "->", SUBMISSION_AGENT.relative_to(REPO_ROOT))

print("\nFirst 500 characters:")
print(SUBMISSION_AGENT.read_text(encoding="utf-8")[:500])


Copied:
src/agents/my_agent.py -> submissions/my_agent.py

First 500 characters:

import numpy as np

try:
    from evaluator.base_agent import BaseAgent
except Exception:
    try:
        from agents.base_agent import BaseAgent
    except Exception:
        class BaseAgent:
            def reset(self): pass
            def seed(self, seed=None): pass


class MyAgent(BaseAgent):
    GRID = 128
    WIND_SIZE = GRID * GRID * 2
    WORLD_SIZE = GRID * GRID

    GOAL = np.array([64.0, 127.0], dtype=np.float32)

    DIRECTIONS = np.array([
        [0, 1],
        [1, 1],
        



# 8. Créer le ZIP final

Très important : `my_agent.py` doit être à la racine du ZIP.

Le ZIP final sera créé ici :

```text
submissions/my_submission.zip
```


In [51]:

ZIP_PATH = SUBMISSIONS_DIR / "my_submission.zip"

if ZIP_PATH.exists():
    ZIP_PATH.unlink()

with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(SUBMISSION_AGENT, arcname="my_agent.py")

print("Created:", ZIP_PATH)
print("Size:", ZIP_PATH.stat().st_size, "bytes")


Created: /home/onyxia/work/stable_v2_RL_sailing_challenge/submissions/my_submission.zip
Size: 2977 bytes



# 9. Vérifier le contenu du ZIP

Il faut voir exactement :

```text
my_agent.py
```

et pas :

```text
submissions/my_agent.py
src/agents/my_agent.py
some_folder/my_agent.py
```


In [52]:

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    names = zf.namelist()

print("ZIP content:")
for name in names:
    print("-", name)

zip_ok = names == ["my_agent.py"]

print("\nZIP STRUCTURE OK =", zip_ok)

if not zip_ok:
    raise RuntimeError("Structure ZIP incorrecte : my_agent.py doit être seul à la racine.")


ZIP content:
- my_agent.py

ZIP STRUCTURE OK = True



# 10. Test d'extraction dans un dossier temporaire

On extrait le ZIP et on vérifie que le fichier existe à la racine.


In [53]:

with tempfile.TemporaryDirectory() as tmpdir:
    tmpdir = Path(tmpdir)

    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(tmpdir)

    extracted_agent = tmpdir / "my_agent.py"

    print("Extracted files:", [p.name for p in tmpdir.iterdir()])
    print("Extracted my_agent.py exists:", extracted_agent.exists())

    if not extracted_agent.exists():
        raise RuntimeError("Après extraction, my_agent.py n'est pas à la racine.")

    extracted_text = extracted_agent.read_text(encoding="utf-8")
    print("Contains class MyAgent:", "class MyAgent" in extracted_text)
    print("Contains act:", "def act" in extracted_text)


Extracted files: ['my_agent.py']
Extracted my_agent.py exists: True
Contains class MyAgent: True
Contains act: True



# 11. Validation optionnelle du fichier extrait

Cette étape copie temporairement le fichier extrait dans `src/agents/tmp_submission_check.py`,
puis lance `test_agent_validity.py`.

Elle évite les mauvaises surprises liées au fichier réellement zippé.


In [54]:

TMP_CHECK_AGENT = AGENTS_DIR / "tmp_submission_check.py"

try:
    shutil.copy2(SUBMISSION_AGENT, TMP_CHECK_AGENT)

    tmp_res = run_command(
        ["python", "test_agent_validity.py", str(TMP_CHECK_AGENT)],
        cwd=SRC_DIR,
        timeout=180,
    )

    print_command_result(tmp_res)

    if tmp_res["returncode"] != 0:
        raise RuntimeError("Le fichier zippé ne passe pas la validation après copie temporaire.")

finally:
    if TMP_CHECK_AGENT.exists():
        TMP_CHECK_AGENT.unlink()


CMD: python test_agent_validity.py /home/onyxia/work/stable_v2_RL_sailing_challenge/src/agents/tmp_submission_check.py
RETURN CODE: 0
ELAPSED: 2.44 sec

STDOUT:
✨ Validating agent in: /home/onyxia/work/stable_v2_RL_sailing_challenge/src/agents/tmp_submission_check.py ✨

🤖 Agent: MyAgent

✅ SUCCESS: Your agent meets all requirements!

⚠️ Warnings (not required to fix, but recommended):
  1. Agent does not implement save() method
  2. Agent does not implement load() method


🎉 Your agent is ready for submission!
Run this command to test performance before submitting:
python src/evaluate_submission.py /home/onyxia/work/stable_v2_RL_sailing_challenge/src/agents/tmp_submission_check.py --seeds 1 --num-seeds 10




# 12. Générer un rapport de soumission

Le rapport est sauvegardé dans :

```text
results/submission_report.json
```


In [55]:

report = {
    "final_agent_source": str(FINAL_AGENT_PATH.relative_to(REPO_ROOT)),
    "submission_agent": str(SUBMISSION_AGENT.relative_to(REPO_ROOT)),
    "zip_path": str(ZIP_PATH.relative_to(REPO_ROOT)),
    "zip_size_bytes": ZIP_PATH.stat().st_size,
    "zip_content": names,
    "static_checks": checks,
    "validity": {
        "returncode": validity_res["returncode"],
        "elapsed_sec": validity_res["elapsed_sec"],
        "stdout_preview": validity_res["stdout"][:2000],
        "stderr_preview": validity_res["stderr"][:2000],
    },
    "local_eval": {
        "run": RUN_LOCAL_EVAL,
        "num_seeds": NUM_SEEDS if RUN_LOCAL_EVAL else None,
        "rows": eval_rows,
        "returncode": eval_res["returncode"] if eval_res else None,
        "stdout_preview": eval_res["stdout"][:4000] if eval_res else None,
        "stderr_preview": eval_res["stderr"][:4000] if eval_res else None,
    },
}

REPORT_PATH = RESULTS_DIR / "submission_report.json"
REPORT_PATH.write_text(json.dumps(report, indent=2), encoding="utf-8")

print("Saved report:", REPORT_PATH)


Saved report: /home/onyxia/work/stable_v2_RL_sailing_challenge/results/submission_report.json



# 13. Dernières consignes avant Codabench

Avant d'uploader :

1. Vérifie que `ZIP STRUCTURE OK = True`.
2. Vérifie que `test_agent_validity.py` passe.
3. Vérifie que l'agent ne fait pas d'entraînement pendant `act`.
4. Vérifie que le fichier ZIP contient uniquement `my_agent.py` à la racine.
5. Upload :

```text
submissions/my_submission.zip
```

Si Codabench échoue :
- regarde d'abord le log ingestion ;
- vérifie que l'import `from evaluator.base_agent import BaseAgent` est présent ou géré par fallback ;
- vérifie qu'aucune dépendance externe non autorisée n'est utilisée.


In [56]:

print("READY TO SUBMIT:")
print(ZIP_PATH)

print("\nTo inspect manually from terminal:")
print(f"unzip -l {ZIP_PATH}")


READY TO SUBMIT:
/home/onyxia/work/stable_v2_RL_sailing_challenge/submissions/my_submission.zip

To inspect manually from terminal:
unzip -l /home/onyxia/work/stable_v2_RL_sailing_challenge/submissions/my_submission.zip
